In [1]:
import pandas as pd

files= ['es.xlsx', 'fr.xlsx', 'de.xlsx', 'gb.xlsx', 'mx.xlsx']
all_countries=[]
for file in files:
    df= pd.read_excel(file)
    df['country']= file.split('.')[0]
    all_countries.append(df)

all= pd.concat(all_countries, ignore_index=True)

all.head()

,liked,topic_detection_liked,disliked,topic_detection_disliked,country
0,Liked · Cercanía al transporte público,hotel-location,"Disliked · Instalaciones antiguas, pese al la...",hotel-atmosphere;hotel-facilities,es
1,Liked · La ubicación y la atención del personal,staff;hotel-location,Disliked · La altura del techo de la habitaci...,room-atmosphere,es
2,Liked · La cama muy cómoda y la cercanía a la...,bed-general;hotel-location,Disliked · Los muebles y la tele un poco anti...,room-atmosphere;room-facilities,es
3,Liked · Tiene muchos restaurantes en el centr...,hotel-facilities,Disliked · La habitación y el cuarto de baño ...,bathroom-general;bathroom-cleanliness;room-gen...,es
4,Liked · El desayuno y la ubicación.,catering-general;hotel-location,Disliked · Camas separadas pese a tener reser...,staff;room-atmosphere,es


In [2]:
liked= all[['liked','topic_detection_liked', 'country']].copy()
liked.columns= ['text', 'labels', 'country']
liked.index=all.index*2

disliked= all[['disliked','topic_detection_disliked', 'country']].copy()
disliked.columns= ['text', 'labels', 'country']
disliked.index=all.index*2+1

final= pd.concat([liked, disliked]).sort_index()

label_list=['hotel-general', 'bathroom-general', 'hotel-atmosphere', 'bathroom-atmosphere',
    'hotel-cleanliness', 'bathroom-cleanliness', 'hotel-facilities', 'bathroom-facilities',
    'hotel-location', 'bed-general', 'hotel-price', 'bed-cleanliness',
    'room-general', 'catering-general', 'room-atmosphere', 'catering-price',
    'room-cleanliness', 'parking', 'room-facilities', 'staff']

binary_result=[]
for labels in final['labels']:
    labels= [l.strip() for l in str(labels).split(';')] if pd.notna(labels) else []
    binary=[]
    for label in label_list:
        if label in labels:
            binary.append(1)
        else:
            binary.append(0)
    binary_result.append(binary)

final[label_list]= pd.DataFrame(binary_result, index=final.index)
final['label_sum']= final[label_list].sum(axis=1)

final.head()

,text,labels,country,hotel-general,bathroom-general,hotel-atmosphere,bathroom-atmosphere,hotel-cleanliness,bathroom-cleanliness,hotel-facilities,...,bed-cleanliness,room-general,catering-general,room-atmosphere,catering-price,room-cleanliness,parking,room-facilities,staff,label_sum
0,Liked · Cercanía al transporte público,hotel-location,es,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,"Disliked · Instalaciones antiguas, pese al la...",hotel-atmosphere;hotel-facilities,es,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,2
2,Liked · La ubicación y la atención del personal,staff;hotel-location,es,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2
3,Disliked · La altura del techo de la habitaci...,room-atmosphere,es,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
4,Liked · La cama muy cómoda y la cercanía a la...,bed-general;hotel-location,es,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2


In [3]:
import re

def clean_and_header(text):
    if not isinstance(text, str):
        return ''
    header=''

    if '·' in text:
        parts= text.split('·', 1)
        header= parts[0].strip().lower()
        text= parts[1].strip()

        if 'disliked' in header:
            header= "[DISLIKED] "
        elif 'liked' in header:
            header= "[LIKED] "
        else:
            return text.replace('·', '').strip()
        text= text

    text= f"{header}{text}"
    text= re.sub(r'http\S+', '', text)
    text= re.sub(r'[^\w\s\-àáâãäåæçèéêëìíîïðñòóôõöùúûüý·\[\]]', ' ', text)
    text= re.sub(r'\s+', ' ', text).strip()

    return text

final['text']= final['text'].apply(clean_and_header)

final= final.drop_duplicates(subset=['text']).reset_index(drop=True)


print(len(final))
print(final[label_list].sum())

1881
hotel-general           246
bathroom-general         35
hotel-atmosphere        298
bathroom-atmosphere      58
hotel-cleanliness       142
bathroom-cleanliness     56
hotel-facilities        204
bathroom-facilities     149
hotel-location          547
bed-general             217
hotel-price             149
bed-cleanliness          29
room-general            131
catering-general        271
room-atmosphere         399
catering-price           50
room-cleanliness        144
parking                  59
room-facilities         299
staff                   644
dtype: int64


In [4]:
final.head()

,text,labels,country,hotel-general,bathroom-general,hotel-atmosphere,bathroom-atmosphere,hotel-cleanliness,bathroom-cleanliness,hotel-facilities,...,bed-cleanliness,room-general,catering-general,room-atmosphere,catering-price,room-cleanliness,parking,room-facilities,staff,label_sum
0,[LIKED] Cercanía al transporte público,hotel-location,es,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,[DISLIKED] Instalaciones antiguas pese al lava...,hotel-atmosphere;hotel-facilities,es,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,2
2,[LIKED] La ubicación y la atención del personal,staff;hotel-location,es,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2
3,[DISLIKED] La altura del techo de la habitació...,room-atmosphere,es,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
4,[LIKED] La cama muy cómoda y la cercanía a la ...,bed-general;hotel-location,es,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2


In [5]:
no_header = final[~final['text'].str.startswith(('[LIKED]', '[DISLIKED]'))]
print(f"no header: {len(no_header)}")

no header: 0


In [6]:
!pip install iterative-stratification

In [7]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np

X= final['text'].values
y= final[label_list].values

msss= MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=41)

for train_index, test_index in msss.split(X,y):
    train_df= final.iloc[train_index].reset_index(drop=True)
    test_df= final.iloc[test_index].reset_index(drop=True)

print((train_df[label_list].mean().sort_values(ascending=False)).head(5))
print((test_df[label_list].mean().sort_values(ascending=False)).head(5))

staff               0.342420
hotel-location      0.290559
room-atmosphere     0.212101
room-facilities     0.158910
hotel-atmosphere    0.158245
dtype: float64
staff               0.342175
hotel-location      0.291777
room-atmosphere     0.212202
hotel-atmosphere    0.159151
room-facilities     0.159151
dtype: float64


In [8]:
def check_progress(df, label_list):
    label_count={}
    for label in label_list:
        country= df[df[label]==1]['country'].value_counts()
        label_count[label]= country
    df= pd.DataFrame(label_count).T.fillna(0).astype(int)
    df['total']=df.sum(axis=1)
    df= df.sort_values('total', ascending=True)
    return df

In [9]:
check_progress(train_df, label_list)

country,de,es,fr,gb,mx,total
bed-cleanliness,1,2,6,7,7,23
bathroom-general,11,2,5,5,5,28
catering-price,11,7,5,14,3,40
bathroom-cleanliness,24,4,7,8,2,45
bathroom-atmosphere,6,13,15,9,3,46
parking,15,13,5,6,8,47
room-general,52,12,16,20,5,105
hotel-cleanliness,29,20,19,25,21,114
room-cleanliness,31,12,20,38,14,115
bathroom-facilities,20,25,28,31,15,119


In [10]:
pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00


In [11]:
import time
import random
from deep_translator import GoogleTranslator
from tqdm import tqdm

def label_augment(df, target_label, goal_counts, threshold=450):
    augment_rows= []
    current= df[df[target_label]== 1].copy()
    current_label_sum= current[label_list].sum(axis=1)

    counts= df[label_list].sum()
    popular_labels= counts[counts>threshold].index.tolist()
    if target_label in popular_labels:
        popular_labels.remove(target_label)

    for lang, goal in goal_counts.items():
        current_num= len(current[current['country']==lang])

        if current_num<goal:
            augment= goal- current_num

            single= current[current_label_sum==1]
            low= current[current_label_sum<=2]

            filtered_single= single[~(single[popular_labels]==1).any(axis=1)]
            filtered_low= low[~(low[popular_labels]==1).any(axis=1)]

            if len(filtered_single)>= augment:
                sources=filtered_single.sample(n=augment, replace=True)
            elif len(filtered_low)> 0:
                sources=filtered_low.sample(n=augment, replace=True)
            else:
                sources= current.sample(n=augment, replace=True)

            for _, row in tqdm(sources.iterrows(), total=len(sources)):
                try:
                    src_text= row['text']
                    src_lang= row['country']
                    dest= 'es' if lang== 'mx' else ('en' if lang=='gb' else lang)
                    if ']' in src_text:
                        divide= src_text.split(']',1)
                        header= divide[0]+']'
                        body= divide[1].strip()
                    else:
                        header=''
                        body= src_text

                    if not body:
                        continue

                    if lang in  ['it', 'nl']:
                        final_text= GoogleTranslator(source='auto', target= lang).translate(body)
                    elif lang=='gb':
                        mid_lang= random.choice(['ko','ja', 'zh-CN'])
                        mid= GoogleTranslator(source='auto', target= mid_lang).translate(body)
                        final_text= GoogleTranslator(source='auto', target= 'en').translate(mid)
                    elif src_lang==lang:
                        mid= GoogleTranslator(source='auto', target= 'en').translate(body)
                        final_text= GoogleTranslator(source='en', target= dest).translate(mid)
                    else:
                        final_text= GoogleTranslator(source='auto', target= dest).translate(body)

                    if final_text and len(final_text.split()) >=3:
                        new= row.copy()
                        new['text']=(header+ " "+final_text).strip()
                        new['country']=lang

                        if current_label_sum[row.name]>1:
                            for l in label_list:
                                if l != target_label:
                                    new[l]=0
                        augment_rows.append(new)

                    time.sleep(0.4)
                except Exception as e:
                    print(f'[ERROR] row {_}, lang {lang}: {e}')
                    time.sleep(2)
                    continue

    if not augment_rows:
        return pd.DataFrame(columns=df.columns)

    augment_df= pd.DataFrame(augment_rows)
    augment_df= augment_df.drop_duplicates(subset=['text']).reset_index(drop=True)
    combine= pd.concat([df[df[target_label]==1],augment_df])
    print(combine['country'].value_counts())

    return augment_df


In [15]:
bed_cleanliness= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
augment_bed_cleanliness= label_augment(train_df, 'bed-cleanliness', bed_cleanliness)
train_df= pd.concat([train_df, augment_bed_cleanliness], ignore_index=True)
train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
train_df.to_csv('bed_cleanliness_backup.csv', index=False)

100%|██████████| 33/33 [00:30<00:00,  1.07it/s]

country
it    41
nl    39
gb    30
fr    29
es    28
de    28
mx    23
Name: count, dtype: int64


In [16]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
bathroom-general,11,2,5,5,0,5,0,28
catering-price,11,7,5,14,0,3,0,40
bathroom-cleanliness,24,4,7,8,0,2,0,45
bathroom-atmosphere,6,13,15,9,0,3,0,46
parking,15,13,5,6,0,8,0,47
room-general,52,12,16,20,0,5,0,105
hotel-cleanliness,29,20,19,25,0,21,0,114
room-cleanliness,31,12,20,38,0,14,0,115
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-price,22,25,27,24,0,21,0,119


In [21]:
bathroom_general= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
augment_bathroom_general= label_augment(train_df, 'bathroom-general',bathroom_general)
train_df= pd.concat([train_df, augment_bathroom_general], ignore_index=True)
train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
train_df.to_csv('bathroom_general_backup.csv', index=False)

100%|██████████| 25/25 [00:21<00:00,  1.15it/s]

country
nl    43
it    42
de    30
gb    29
fr    29
es    28
mx    18
Name: count, dtype: int64


In [22]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
catering-price,11,7,5,14,0,3,0,40
bathroom-cleanliness,24,4,7,8,0,2,0,45
bathroom-atmosphere,6,13,15,9,0,3,0,46
parking,15,13,5,6,0,8,0,47
room-general,52,12,16,20,0,5,0,105
hotel-cleanliness,29,20,19,25,0,21,0,114
room-cleanliness,31,12,20,38,0,14,0,115
hotel-price,22,25,27,24,0,21,0,119
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-facilities,25,33,33,44,0,28,0,163


In [24]:
catering_price= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
augment_catering_price= label_augment(train_df, 'catering-price',catering_price)
train_df= pd.concat([train_df, augment_catering_price], ignore_index=True)
train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
train_df.to_csv('catering_price_backup.csv', index=False)

100%|██████████| 35/35 [00:32<00:00,  1.06it/s]

country
it    39
nl    36
de    30
gb    30
fr    29
es    27
mx    20
Name: count, dtype: int64


In [25]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
bathroom-cleanliness,24,4,7,8,0,2,0,45
bathroom-atmosphere,6,13,15,9,0,3,0,46
parking,15,13,5,6,0,8,0,47
room-general,52,12,16,20,0,5,0,105
hotel-cleanliness,29,20,19,25,0,21,0,114
room-cleanliness,31,12,20,38,0,14,0,115
hotel-price,22,25,27,24,0,21,0,119
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-facilities,25,33,33,44,0,28,0,163
bathroom-general,27,20,26,27,28,13,31,172


In [29]:
bathroom_cleanliness= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
augment_bathroom_cleanliness= label_augment(train_df, 'bathroom-cleanliness',bathroom_cleanliness)
train_df= pd.concat([train_df, augment_bathroom_cleanliness], ignore_index=True)
train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
train_df.to_csv('bathroom_cleanliness_backup.csv', index=False)

100%|██████████| 26/26 [00:28<00:00,  1.09s/it]

country
it    47
nl    45
es    30
de    30
fr    30
gb    30
mx    22
Name: count, dtype: int64


In [32]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
bathroom-atmosphere,6,13,15,9,0,3,0,46
parking,15,13,5,6,0,8,0,47
room-general,52,12,16,20,0,5,0,105
hotel-cleanliness,29,20,19,25,0,21,0,114
room-cleanliness,31,12,20,38,0,14,0,115
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-price,22,25,27,24,0,21,0,119
hotel-facilities,25,33,33,44,0,28,0,163
bathroom-general,27,20,26,27,28,13,31,172
bed-general,34,48,33,37,0,22,0,174


In [35]:
bathroom_atmosphere= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
augment_bathroom_atmosphere= label_augment(train_df, 'bathroom-atmosphere',bathroom_atmosphere)
train_df= pd.concat([train_df, augment_bathroom_atmosphere], ignore_index=True)
train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
train_df.to_csv('bathroom_atmosphere_backup.csv', index=False)

100%|██████████| 42/42 [00:40<00:00,  1.04it/s]

country
nl    32
gb    30
es    30
it    30
fr    28
de    27
mx    16
Name: count, dtype: int64


In [36]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
parking,15,13,5,6,0,8,0,47
room-general,52,12,16,20,0,5,0,105
hotel-cleanliness,29,20,19,25,0,21,0,114
room-cleanliness,31,12,20,38,0,14,0,115
hotel-price,22,25,27,24,0,21,0,119
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-facilities,25,33,33,44,0,28,0,163
bathroom-atmosphere,21,30,26,30,22,11,25,165
bathroom-general,27,20,26,27,28,13,31,172
bed-general,34,48,33,37,0,22,0,174


In [39]:
parking= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
augment_parking= label_augment(train_df, 'parking',parking)
train_df= pd.concat([train_df, augment_parking], ignore_index=True)
train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
train_df.to_csv('parking_backup.csv', index=False)

100%|██████████| 33/33 [00:31<00:00,  1.04it/s]

country
it    45
nl    40
es    30
gb    30
fr    29
de    29
mx    22
Name: count, dtype: int64


In [41]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
room-general,52,12,16,20,0,5,0,105
hotel-cleanliness,29,20,19,25,0,21,0,114
room-cleanliness,31,12,20,38,0,14,0,115
hotel-price,22,25,27,24,0,21,0,119
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-facilities,25,33,33,44,0,28,0,163
bathroom-atmosphere,21,30,26,30,22,11,25,165
bathroom-general,27,20,26,27,28,13,31,172
bed-general,34,48,33,37,0,22,0,174
catering-price,27,23,24,30,30,14,29,177


In [42]:
for label in ['room-general', 'hotel-cleanliness', 'room-cleanliness']:
    goal= {'de': 30, 'es':30, 'fr':30, 'gb':30, 'mx':30, 'it':50, 'nl':50}
    augment= label_augment(train_df, label, goal)
    if not augment.empty:
        train_df= pd.concat([train_df, augment], ignore_index=True)
        train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
        train_df.to_csv(f'{label}_backup.csv', index=False)
    check_progress(train_df, label_list)

100%|██████████| 50/50 [00:41<00:00,  1.21it/s]


country
de    52
gb    28
fr    26
es    23
it    23
nl    19
mx    13
Name: count, dtype: int64


100%|██████████| 50/50 [00:40<00:00,  1.23it/s]


country
de    29
gb    28
fr    22
mx    22
es    21
it    19
nl    15
Name: count, dtype: int64


100%|██████████| 50/50 [00:41<00:00,  1.21it/s]

country
gb    38
de    31
es    28
fr    26
nl    24
it    22
mx    21
Name: count, dtype: int64


In [43]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
bathroom-facilities,20,25,28,31,0,15,0,119
hotel-price,22,25,27,24,0,21,0,119
hotel-cleanliness,29,21,22,28,19,22,15,156
hotel-facilities,25,33,33,44,0,28,0,163
bathroom-atmosphere,21,30,26,30,22,11,25,165
bathroom-general,27,20,26,27,28,13,31,172
bed-general,34,48,33,37,0,22,0,174
room-cleanliness,31,23,26,38,18,18,21,175
catering-price,27,23,24,30,30,14,29,177
room-general,52,21,26,28,23,12,19,181


In [46]:
for label in label_list:
    it_count= len(train_df[(train_df[label]==1)& (train_df['country']=='it')])
    nl_count= len(train_df[(train_df[label]==1)& (train_df['country']=='nl')])
    if it_count<25 or nl_count<25:
        goal= {'it':50, 'nl':50}
        augment= label_augment(train_df, label, goal)

        if not augment.empty:
            train_df= pd.concat([train_df, augment], ignore_index=True)
            train_df= train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
            train_df.to_csv(f'{label}_backup.csv', index=False)

100%|██████████| 33/33 [00:20<00:00,  1.63it/s]


country
de    50
fr    43
gb    43
es    40
it    36
nl    34
mx    21
Name: count, dtype: int64


100%|██████████| 26/26 [00:25<00:00,  1.03it/s]


country
nl    40
it    40
de    29
gb    28
fr    22
mx    22
es    21
Name: count, dtype: int64


100%|██████████| 32/32 [00:31<00:00,  1.02it/s]


...

country
nl    36
it    36
fr    27
es    25
gb    24
de    22
mx    21
Name: count, dtype: int64


100%|██████████| 25/25 [00:22<00:00,  1.09it/s]


country
de    71
gb    46
it    44
nl    44
fr    37
es    32
mx    31
Name: count, dtype: int64


100%|██████████| 30/30 [00:33<00:00,  1.12s/it]

country
gb    65
fr    49
es    46
de    45
it    45
nl    41
mx    34
Name: count, dtype: int64


In [47]:
check_progress(train_df, label_list)

country,de,es,fr,gb,it,mx,nl,total
bathroom-facilities,20,25,28,31,23,15,27,169
hotel-price,22,25,27,24,25,21,26,170
bathroom-general,27,20,26,27,28,13,31,172
catering-price,27,23,24,30,30,14,29,177
hotel-cleanliness,29,21,22,28,29,22,28,179
bed-cleanliness,23,19,25,30,34,21,29,181
bathroom-atmosphere,21,30,26,30,30,11,34,182
parking,27,29,26,30,33,18,29,192
room-cleanliness,31,23,26,38,29,18,30,195
room-general,52,21,26,28,33,12,34,206


In [48]:
train_df= train_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df[['text']+label_list].to_csv('train_data.csv', index=False)
train_df[['text', 'country']+label_list].to_csv('train_data_with_country.csv', index=False)

test_df[['text']+label_list].to_csv('val_data.csv', index=False)
test_df[['text', 'country']+label_list].to_csv('val_data_with_country.csv', index=False)